In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

path = "data/"
print(os.listdir(path))

dfs = []
for f in sorted(os.listdir(path)):
    if f.endswith(".csv"):
        full_path = os.path.join(path, f)
        print(f"Path: {full_path}")
        dfs.append(pd.read_csv(full_path))

df = pd.concat(dfs, ignore_index=True)
print(f"Total: {len(df):,}")

df["tpep_pickup_datetime"]  = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])
df["trip_duration"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds()

df = df[(df["trip_duration"] >= 60) & (df["trip_duration"] <= 10800)]
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]
df = df[(df["pickup_longitude"] >= -74.25) & (df["pickup_longitude"] <= -73.70)]
df = df[(df["pickup_latitude"]  >= 40.48)  & (df["pickup_latitude"]  <= 40.92)]
df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 6)]
df = df.dropna()
print(f"Cleaned Total: {len(df):,}")

df["pickup_hour"]  = df["tpep_pickup_datetime"].dt.hour
df["day_of_week"]  = df["tpep_pickup_datetime"].dt.dayofweek
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["is_rush_hour"] = df["pickup_hour"].isin([7,8,9,17,18,19]).astype(int)

FEATURES = ["trip_distance", "pickup_hour", "day_of_week",
            "passenger_count", "is_rush_hour", "is_weekend"]

X = df[FEATURES]
y = df["trip_duration"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
mae   = mean_absolute_error(y_test, y_pred)
r2    = r2_score(y_test, y_pred)
rmsle = np.sqrt(np.mean((np.log1p(np.maximum(y_pred,0)) - np.log1p(y_test))**2))

print(f"\nRMSE  : {rmse:.1f} sn ({rmse/60:.1f} dk)")
print(f"MAE   : {mae:.1f} sn ({mae/60:.1f} dk)")
print(f"R²    : {r2:.4f}")
print(f"RMSLE : {rmsle:.4f}")

print("\nConstants:")
for f, c in zip(FEATURES, model.coef_):
    print(f"  {f:<20}: {c:+.3f}")

In [ ]:
!pip install h3
import h3
import folium
import pandas as pd

RESOLUTION = 9

df["pickup_h3"] = [
    h3.latlng_to_cell(lat, lng, RESOLUTION)
    for lat, lng in zip(df["pickup_latitude"], df["pickup_longitude"])
]

cell_stats = df.groupby("pickup_h3").size().reset_index(name="trip_count")

print(f"Total H3 Cells: {len(cell_stats)}")
print(cell_stats.sort_values("trip_count", ascending=False).head())

import numpy as np

max_log = np.log1p(cell_stats["trip_count"].max())

def get_color(count):
    ratio = np.log1p(count) / max_log
    if ratio > 0.85: return "#800026"
    elif ratio > 0.70: return "#BD0026"
    elif ratio > 0.55: return "#E31A1C"
    elif ratio > 0.40: return "#FC4E2A"
    elif ratio > 0.25: return "#FD8D3C"
    elif ratio > 0.10: return "#FEB24C"
    else:              return "#FED976"

m = folium.Map(location=[40.75, -73.99], zoom_start=12)

for _, row in cell_stats.iterrows():
    boundary = h3.cell_to_boundary(row["pickup_h3"])
    coords = [[lat, lng] for lat, lng in boundary]
    color = get_color(row["trip_count"])

    folium.Polygon(
        locations=coords,
        color=color,
        weight=0.2,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        tooltip=f"Yolculuk: {row['trip_count']}"
    ).add_to(m)

legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px; border-radius:8px;
     font-size:13px; box-shadow:2px 2px 6px rgba(0,0,0,0.3)">
  <b>Yolculuk Yoğunluğu</b><br>
  <i style="background:#800026;width:14px;height:14px;display:inline-block"></i> High<br>
  <i style="background:#FC4E2A;width:14px;height:14px;display:inline-block"></i> Mid<br>
  <i style="background:#FED976;width:14px;height:14px;display:inline-block"></i> Low<br>
  <small>Renk skalası log-normalize edilmiştir.</small>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("h3_map.html")